# M1 OOP Measurement Model Fit

Objective:
- Re-fit the FACE M1 measurement model with the parallel OOP implementation.
- Start with smoke checks, then run cached staged fits when diagnostics are acceptable.
- Produce diagnostics, core measurement visualizations, and patient projection examples.

Success criteria:
- No imputation is introduced; all fits use observed-cell likelihoods.
- Soft unlikely cells and near-zero G-anchor-specific priors are active unless `fast_mode=True` is selected.
- Stage diagnostics are recorded after each fit before moving to longer runs.


In [5]:
# Setup: imports, paths, and reproducibility
from __future__ import annotations

from pathlib import Path
import sys
import time

import arviz as az
import numpy as np
import pandas as pd

# Notebook kernels may start in notebooks/; locate the repository root explicitly.
START = Path.cwd().resolve()
for candidate in (START, *START.parents):
    if (candidate / "src" / "face" / "models").exists():
        REPO = candidate
        break
else:
    raise RuntimeError(f"Could not locate FACE repository root from {START}")

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

loaded_face = sys.modules.get("face")
loaded_face_file = getattr(loaded_face, "__file__", None) if loaded_face is not None else None
if loaded_face is not None and (
    loaded_face_file is None or SRC not in Path(loaded_face_file).resolve().parents
):
    for module_name in [name for name in sys.modules if name == "face" or name.startswith("face.")]:
        del sys.modules[module_name]

from face.models.bayesian.measurement_model_oop import (
    MeasurementConfig,
    MeasurementDataset,
    StageRunner,
    PatientProjector,
    MeasurementVisualizer,
    S1_FACTORS,
)

SEED = 20260605
np.random.seed(SEED)
REPO


PosixPath('/Users/andriikulakovskyi/Desktop/face-common-bp-sz-dr')

## Run Modes

Use `RUN_MODE = "smoke"` first. Smoke mode is a wiring check, not convergence evidence.

- `smoke`: fast S1-only fit using the speed-oriented config; validates imports, masks, model construction, caching, and plotting.
- `medium`: balanced subsample for diagnostic development with the theory-faithful config.
- `production`: full continuous stages and largest-N mixed certification; expected to be compute-heavy.

Set `INCLUDE_MIXED_SMOKE = True` only when you explicitly want to test mixed-likelihood construction. The mixed smoke run is too short for scientific diagnostics.

Caches are reused only when their model version and stage signature match the current implementation.


In [6]:
# Configure the OOP measurement model
RUN_MODE = "smoke"  # one of: smoke, medium, production
FAST_MODE = False    # True mirrors the old hard-zero complement for optimization triage
INCLUDE_MIXED_SMOKE = False

base_config = MeasurementConfig()
if RUN_MODE == "smoke":
    # Smoke is a wiring check, so it uses a small fast config by default.
    # Medium/production below use the theory-faithful defaults unless FAST_MODE=True.
    config = base_config.with_smoke_defaults()
    stages = list(config.smoke_stage_plan)
    if INCLUDE_MIXED_SMOKE:
        stages += config.mixed_smoke_stage_plan
elif RUN_MODE == "medium":
    config = base_config.with_fast_mode() if FAST_MODE else base_config
    stages = [
        stage.__class__(**{**stage.__dict__, "n_subsample": 2000, "balanced": True, "draws": 500, "tune": 700, "chains": 4})
        for stage in config.stage_plan
    ]
elif RUN_MODE == "production":
    config = base_config.with_fast_mode() if FAST_MODE else base_config
    stages = config.stage_plan
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

runner = StageRunner(config)
dataset = MeasurementDataset(config)
projector = PatientProjector(config)
visualizer = MeasurementVisualizer(config)

[(s.name, s.mixed, s.n_subsample, s.draws, s.tune, s.chains, s.target_accept) for s in stages]


[('smoke_s1_wiring_v2', False, 300, 300, 500, 4, 0.95)]

## Data And Prior Checks

These checks summarize the exact processed inputs used by the model. They do not modify data.


In [7]:
# Input summary: same artifacts as the current measurement engine
baseline_path = config.processed_dir / "baseline_v0.parquet"
prior_path = config.prior_matrix
summary_n = stages[0].n_subsample if RUN_MODE == "smoke" else None

baseline = pd.read_parquet(baseline_path)
prior = pd.read_csv(prior_path)
core = dataset.core(S1_FACTORS, n_subsample=summary_n, seed=SEED)
spec = dataset.loading_spec(core, windows=False)

summary = {
    "baseline_shape": baseline.shape,
    "baseline_missing_fraction": float(baseline.isna().mean().mean()),
    "prior_cells": len(prior),
    "core_shape": core.M.shape,
    "free_loading_cells": spec.n_free,
    "covariates": core.covariate_names,
    "mode_note": "smoke uses fast wiring config" if RUN_MODE == "smoke" else "theory-faithful config unless FAST_MODE=True",
}
summary


{'baseline_shape': (9013, 143),
 'baseline_missing_fraction': 0.39774715465384497,
 'prior_cells': 1430,
 'core_shape': (300, 68),
 'free_loading_cells': 125,
 'covariates': [],
 'mode_note': 'smoke uses fast wiring config'}

## Staged Fit

Run the cell below after the data/prior checks look correct. In smoke mode this should be short; in production mode it can take hours. Every stage writes an `idata.nc` and `manifest.json` cache under `results/face/oop_measurement/<stage>/`.


In [8]:
# Run staged fits with cache reuse
stage_results = {}
for stage in stages:
    t0 = time.time()
    idata, manifest = runner.run_stage(stage, overwrite=False)
    stage_results[stage.name] = {"idata": idata, "manifest": manifest, "elapsed_sec": time.time() - t0}
    print(stage.name, manifest.get("diagnostics", {}), f"elapsed={stage_results[stage.name]['elapsed_sec']:.1f}s")

list(stage_results)

smoke_s1_wiring_v2 {'rhat': 1.02, 'ess': 252.0, 'divergences': 0} elapsed=0.0s


['smoke_s1_wiring_v2']

## Diagnostics Gate

Review this table before scaling up.

- Smoke mode: validates the path runs and writes a manifest; do not interpret R-hat/ESS as scientific evidence.
- Medium/production: require no divergences and acceptable R-hat/ESS for structural parameters before trusting the fit.

If a cached stage has no current `model_version` or a stale stage signature, the runner refits it rather than silently reusing old smoke results.

In [9]:
# Diagnostics table from manifests
rows = []
for name, result in stage_results.items():
    manifest = result["manifest"]
    diag = manifest.get("diagnostics", {})
    rows.append({
        "stage": name,
        "N": manifest.get("N"),
        "J": manifest.get("J"),
        "rhat": diag.get("rhat"),
        "ess": diag.get("ess"),
        "divergences": diag.get("divergences"),
        "elapsed_sec": round(result["elapsed_sec"], 1),
    })

diagnostics = pd.DataFrame(rows)
diagnostics

,stage,N,J,rhat,ess,divergences,elapsed_sec
0,smoke_s1_wiring_v2,300,68,1.02,252.0,0,0.0


## Core Visualizations

The next cells use the latest fitted stage in `stage_results`. For a publication-grade atlas, run production or medium fits first; smoke figures only verify plotting and data flow.

In [10]:
# Select a fitted continuous stage for loading/Phi/projection examples
continuous_stage = next(s for s in reversed(stages) if not s.mixed)
continuous_result = stage_results[continuous_stage.name]
continuous_idata = continuous_result["idata"]

plot_core = dataset.core(
    continuous_stage.factors,
    correlated=continuous_stage.correlated,
    windows=continuous_stage.windows,
    n_subsample=continuous_stage.n_subsample if RUN_MODE == "smoke" else None,
    seed=continuous_stage.seed,
)
plot_spec = dataset.loading_spec(plot_core, windows=continuous_stage.windows)

atlas_path = visualizer.loading_atlas(plot_spec, continuous_idata.posterior, filename=f"{continuous_stage.name}_loading_atlas.png")
phi_path = visualizer.phi_heatmap(continuous_idata.posterior, plot_core.factor_cols, filename=f"{continuous_stage.name}_phi.png")

atlas_path, phi_path

(PosixPath('/Users/andriikulakovskyi/Desktop/face-common-bp-sz-dr/docs/figures/oop_measurement/smoke_s1_wiring_v2_loading_atlas.png'),
 PosixPath('/Users/andriikulakovskyi/Desktop/face-common-bp-sz-dr/docs/figures/oop_measurement/smoke_s1_wiring_v2_phi.png'))

## Patient Projection Examples

The examples below project patients onto the fitted continuous map using only observed cells. They intentionally choose one patient per reliability pattern where available: well-characterized, partial, and prior-dominated.

In [11]:
# Compute coordinate means, intervals, and reliability tiers
projection = projector.projection_frame(plot_core, continuous_idata.posterior)
reliability_path = visualizer.reliability_bar(projection, plot_core.factor_cols, filename=f"{continuous_stage.name}_reliability.png")

# Pick example patients by overall severity reliability when possible
rel_col = "overall_severity__reliability"
examples = {}
for tier in ["well", "partial", "prior-dominated"]:
    matches = projection.index[projection[rel_col] == tier]
    if len(matches):
        examples[tier] = matches[0]

examples, reliability_path

({'well': ('bp', '10333'),
  'partial': ('bp', '42020'),
  'prior-dominated': ('bp', '10462')},
 PosixPath('/Users/andriikulakovskyi/Desktop/face-common-bp-sz-dr/docs/figures/oop_measurement/smoke_s1_wiring_v2_reliability.png'))

In [12]:
# Plot patient position and uncertainty examples
patient_figures = {}
for tier, patient_id in examples.items():
    filename = f"{continuous_stage.name}_patient_{tier}.png".replace("/", "_")
    patient_figures[tier] = visualizer.patient_uncertainty(
        projection,
        patient_id,
        plot_core.factor_cols,
        filename=filename,
    )

projection.loc[list(examples.values())].filter(regex="__(mean|sd|reliability)$"), patient_figures

(                   overall_severity__mean  overall_severity__sd  \
 cohort patient_id                                                 
 bp     10333                     0.732317              0.101411   
        42020                    -1.211271              0.738697   
        10462                     0.000000              1.000000   
 
                   overall_severity__reliability  cognition__mean  \
 cohort patient_id                                                  
 bp     10333                               well        -1.211939   
        42020                            partial         0.253064   
        10462                    prior-dominated         0.000000   
 
                    cognition__sd cognition__reliability  metabolic__mean  \
 cohort patient_id                                                          
 bp     10333            0.203403                   well         0.172115   
        42020            0.235867                   well         0.714112   
   

In [ ]:
# 2D patient-projection map: biology (metabolic) vs general severity (G), with the
# example patients drawn as 94% HDI crosses. Shows both *position* and *uncertainty*
# on the measurement map. Biology should sit roughly orthogonal to G (bifactor).
map_path = visualizer.map_scatter(
    projection,
    "overall_severity",
    "metabolic",
    highlight=examples,
    filename=f"{continuous_stage.name}_map_scatter.png",
)
map_path

## Scale-Up Checklist

Before switching from `smoke` to `medium` or `production`:

- Confirm the input summary uses the expected patient and indicator counts.
- Confirm `free_loading_cells` is higher than the old hard-zero engine when `FAST_MODE=False`.
- Confirm smoke stages write manifests and have no unexpected model-construction errors.
- Review divergences, R-hat, ESS, and elapsed time.
- Inspect atlas, Phi, reliability, and patient uncertainty figures.
- Only then increase `RUN_MODE`; keep cache reuse enabled.